# บทเรียน 09 - รูปแบบการออกแบบเมตาคอกนิชัน


## การตั้งค่า

โน้ตบุ๊กนี้สาธิตรูปแบบการออกแบบ Metacognition โดยใช้ Microsoft Agent Framework.

**ข้อกำหนดเบื้องต้น:**
- การปรับใช้ Azure OpenAI ที่กำหนดค่าผ่านตัวแปรสภาพแวดล้อม
- Azure CLI ได้รับการตรวจสอบสิทธิ์ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## เมตาคอกนิชันคืออะไร?

เมตาคอกนิชันคือ **การคิดเกี่ยวกับการคิด**. ในบริบทของตัวแทนปัญญาประดิษฐ์ หมายถึงการสร้างตัวแทนที่สามารถ:

- **ไตร่ตรองตนเอง** เกี่ยวกับผลลัพธ์และกระบวนการให้เหตุผลของตัวเอง
- **ตรวจจับข้อผิดพลาด** และกู้คืนอย่างราบรื่น แทนที่จะล้มเหลวโดยเงียบๆ
- **ประเมิน** ว่าคำตอบของพวกเขาครบถ้วนและเป็นประโยชน์หรือไม่
- **ปรับกลยุทธ์** เมื่อแนวทางเริ่มต้นใช้ไม่ได้ (เช่น หันไปใช้ระบบสำรอง)

ตัวแทนที่มีความสามารถเมตาคอกนิชันไม่ได้มีเพียงแค่ตอบคำถาม — แต่ยังเฝ้าติดตามการทำงานของตัวเองและปรับตัวได้ทันที


## เครื่องมือหลักและสำรอง

รูปแบบการคิดเชิงเมตา (metacognition) ที่พบบ่อยคือ **กลยุทธ์การสำรอง (fallback strategy)**. ตัวแทนจะลองใช้เครื่องมือหลักก่อน; หากล้มเหลว (เช่น ข้อผิดพลาด 404) ตัวแทนจะตรวจพบความล้มเหลวและสลับไปใช้เครื่องมือสำรองอย่างโปร่งใส

สิ่งนี้สะท้อนถึงระบบในโลกจริงที่บริการหลักอาจไม่สามารถใช้งานได้ และตัวแทนต้องวินิจฉัยปัญหาด้วยตนเองก่อนที่จะเลือกเส้นทางทางเลือก

ด้านล่างเรากำหนดเครื่องมือค้นหาเที่ยวบินสองอย่าง:
- **หลัก** — ครอบคลุม Paris, Tokyo, และ Barcelona
- **สำรอง** — ครอบคลุม Berlin, Sydney, และ New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ตัวแทนสะท้อนตนเองพร้อมการกู้คืนจากข้อผิดพลาด

ตัวแทนด้านล่างได้รับคำสั่งให้ลองใช้ระบบการบินหลักก่อน ตรวจจับความล้มเหลว และถอยกลับไปใช้ระบบสำรองอย่างโปร่งใส หลังจากแต่ละการตอบ มันจะประเมินตนเองโดยย่อว่าได้ตอบคำถามของผู้ใช้ครบถ้วนหรือไม่


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## แบบแผนการประเมินตนเอง

อีกด้านหนึ่งของการรับรู้เหนือความคิด (metacognition) คือ **การประเมินตนเอง**: ตัวแทนแยกต่างหาก (หรืออาจเป็นตัวแทนเดียวกันในการตรวจทวนครั้งที่สอง) จะทบทวนคำตอบเพื่อความสมบูรณ์ ความถูกต้อง และความเป็นประโยชน์

ด้านล่างเราสร้างตัวแทน `ResponseEvaluator` ที่ให้คะแนนคำตอบของตัวแทนท่องเที่ยวในสามมิติ


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีสร้าง **เอเจนต์ที่รู้เท่าทันการคิดของตนเอง (metacognitive agents)** โดยใช้ Microsoft Agent Framework:

- **การสะท้อนตนเอง**: เอเจนต์ที่ตรวจสอบการให้เหตุผลของตัวเองและสื่อสารอย่างโปร่งใสว่าเกิดอะไรขึ้น
- **การกู้คืนจากข้อผิดพลาดโดยมีตัวสำรอง**: รูปแบบเครื่องมือหลัก + สำรองที่เอเจนต์ตรวจจับความล้มเหลว (เช่น ข้อผิดพลาด 404) และจะพยายามใช้แหล่งข้อมูลทางเลือกโดยอัตโนมัติ
- **การประเมินตนเอง**: เอเจนต์ผู้ประเมินแยกต่างหากที่ให้คะแนนคำตอบในด้านความสมบูรณ์ ความถูกต้อง และความเป็นประโยชน์

รูปแบบเหล่านี้ช่วยทำให้เอเจนต์มีความทนทาน โปร่งใส และน่าเชื่อถือมากขึ้น — คุณสมบัติสำคัญสำหรับการนำไปใช้งานจริง


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ข้อจำกัดความรับผิดชอบ:
เอกสารฉบับนี้ถูกแปลโดยใช้บริการแปลภาษาด้วยปัญญาประดิษฐ์ Co-op Translator (https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้การแปลถูกต้อง โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นฉบับควรถือเป็นแหล่งข้อมูลที่เชื่อถือได้ที่สุด สำหรับข้อมูลที่สำคัญ ขอแนะนำให้ใช้บริการแปลโดยนักแปลมืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใดๆ ที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
